# INV-M365-CK — Persona-Action Mapping Audit v1

## Purpose
Freeze the `G2` mapping/orphan/stub audit for the workforce graph.

## Lemma Mapping
- `L86_m365_persona_action_mapping_audit_v1`

## Invariant Support
- `invariants/lemmas/L86_m365_persona_action_mapping_audit_v1.yaml`

## Expected Result
- `184` unique persona-facing aliases fully classified
- `430` active persona/action pairs fully classified
- `84` reverse direct orphans published


In [ ]:
from __future__ import annotations
import ast
import json
from collections import defaultdict
from pathlib import Path
import sys
import yaml

repo = Path.cwd()
if not (repo / "registry/persona_registry_v2.yaml").exists():
    repo = Path("/Users/smarthaus/Projects/GitHub/M365")
sys.path.insert(0, str(repo / "src"))
from smarthaus_common.executor_routing import executor_route_for_action

root = repo / "registry"
preg = yaml.safe_load((root / "persona_registry_v2.yaml").read_text())
direct = json.loads((repo / "artifacts/diagnostics/m365_direct_full_surface_certification.json").read_text())
source = (repo / "src/ops_adapter/actions.py").read_text()
mod = ast.parse(source)
func = next(node for node in mod.body if isinstance(node, ast.AsyncFunctionDef) and node.name == "_execute_impl")

def parse_test(expr):
    agents = None
    actions = None
    if isinstance(expr, ast.Compare) and isinstance(expr.left, ast.Name) and len(expr.ops) == 1:
        name = expr.left.id
        comp = expr.comparators[0]
        vals = []
        if isinstance(comp, ast.Constant) and isinstance(comp.value, str):
            vals = [comp.value]
        elif isinstance(comp, ast.Tuple):
            vals = [elt.value for elt in comp.elts if isinstance(elt, ast.Constant) and isinstance(elt.value, str)]
        if isinstance(expr.ops[0], (ast.Eq, ast.In)) and vals:
            if name == "agent":
                agents = vals
            elif name == "action":
                actions = vals
    elif isinstance(expr, ast.BoolOp) and isinstance(expr.op, ast.And):
        for value in expr.values:
            a, b = parse_test(value)
            if a:
                agents = a if agents is None else sorted(set(agents) | set(a))
            if b:
                actions = b if actions is None else sorted(set(actions) | set(b))
    return agents, actions

def body_profile(stmts):
    body_has_await = any(isinstance(n, ast.Await) for stmt in stmts for n in ast.walk(stmt))
    first_return_kind = "none"
    for stmt in stmts:
        if isinstance(stmt, ast.Return):
            val = stmt.value
            if isinstance(val, ast.Await):
                first_return_kind = "await"
            elif isinstance(val, ast.Call):
                first_return_kind = "call"
            elif isinstance(val, (ast.Dict, ast.List, ast.Constant, ast.JoinedStr)):
                first_return_kind = "literal"
            elif isinstance(val, ast.Name):
                first_return_kind = "name"
            else:
                first_return_kind = type(val).__name__
            break
    return body_has_await, first_return_kind

rows = []
def walk(nodes, current_agents=None):
    for node in nodes:
        if isinstance(node, ast.If):
            agents, actions = parse_test(node.test)
            next_agents = current_agents
            if agents and not actions:
                next_agents = agents
            if actions and (agents or current_agents):
                final_agents = agents or current_agents or []
                body_has_await, first_return_kind = body_profile(node.body)
                rows.append({
                    "agents": list(final_agents),
                    "actions": list(actions),
                    "body_has_await": body_has_await,
                    "first_return_kind": first_return_kind,
                })
            walk(node.body, next_agents)
            walk(node.orelse, current_agents)
walk(func.body)

explicit_pairs = set()
pure_literal_pairs = set()
for row in rows:
    for agent in row["agents"]:
        for action in row["actions"]:
            explicit_pairs.add((agent, action))
            if row["first_return_kind"] == "literal" and not row["body_has_await"]:
                pure_literal_pairs.add((agent, action))

alias_map = defaultdict(set)
direct_actions = set()
for path in root.glob("*expansion_v2.yaml"):
    data = yaml.safe_load(path.read_text()) or {}
    for action, meta in (data.get("supported_actions") or {}).items():
        direct_actions.add(str(action))
        if isinstance(meta, dict) and meta.get("agent_alias"):
            alias_map[str(meta["agent_alias"])] .add(str(action))
recipes = yaml.safe_load((root / "cross_workload_automation_recipes_v2.yaml").read_text()) or {}
for action, meta in (recipes.get("catalog_actions") or {}).items():
    direct_actions.add(str(action))
    if isinstance(meta, dict) and meta.get("agent_alias"):
        alias_map[str(meta["agent_alias"])] .add(str(action))

known_stub_pairs = {(agent, action) for agent, actions in direct["legacy_stub_inventory"]["by_agent"].items() for action in actions}
expanded_stub_pairs = known_stub_pairs | pure_literal_pairs
non_stub_explicit_pairs = explicit_pairs - expanded_stub_pairs

pair_status = {}
unique_owners = defaultdict(lambda: {"active": [], "planned": []})
active_alias_to_direct = defaultdict(set)
for pid, persona in preg["personas"].items():
    status = persona["status"]
    for action in persona.get("allowed_actions", []):
        unique_owners[action][status].append(pid)
        if status != "active":
            pair_status[(pid, action)] = "fenced"
            continue
        if action in alias_map:
            pair_status[(pid, action)] = "mapped"
            active_alias_to_direct[action].update(alias_map[action])
            continue
        if (pid, action) in expanded_stub_pairs:
            pair_status[(pid, action)] = "legacy-stubbed"
            continue
        if (pid, action) in non_stub_explicit_pairs:
            pair_status[(pid, action)] = "mapped"
            continue
        try:
            executor_route_for_action(pid, action)
        except Exception:
            pair_status[(pid, action)] = "orphaned"
        else:
            pair_status[(pid, action)] = "dead-routed"

pair_counts = defaultdict(int)
for status in pair_status.values():
    pair_counts[status] += 1

unique_class = {}
for action, owners in unique_owners.items():
    active = owners["active"]
    if not active:
        unique_class[action] = "fenced"
        continue
    statuses = {pair_status[(pid, action)] for pid in active}
    if "mapped" in statuses:
        unique_class[action] = "mapped"
    elif "legacy-stubbed" in statuses:
        unique_class[action] = "legacy-stubbed"
    elif "dead-routed" in statuses:
        unique_class[action] = "dead-routed"
    elif "orphaned" in statuses:
        unique_class[action] = "orphaned"
    else:
        unique_class[action] = sorted(statuses)[0]

unique_counts = defaultdict(int)
for status in unique_class.values():
    unique_counts[status] += 1

owned_direct = set()
for alias, directs in active_alias_to_direct.items():
    owned_direct.update(directs)
active_aliases = {action for persona in preg["personas"].values() if persona["status"] == "active" for action in persona.get("allowed_actions", [])}
owned_direct.update(active_aliases & direct_actions)
direct_orphans = sorted(direct_actions - owned_direct)

mixed_aliases = []
for action, owners in unique_owners.items():
    active = owners["active"]
    statuses = sorted({pair_status[(pid, action)] for pid in active}) if active else ["fenced"]
    if len(statuses) > 1:
        mixed_aliases.append({"action": action, "statuses": statuses, "owners": active})

summary = {
    "unique_action_classification": dict(sorted(unique_counts.items())),
    "pair_classification": dict(sorted(pair_counts.items())),
    "reverse_direct_orphans_total": len(direct_orphans),
    "mixed_aliases_total": len(mixed_aliases),
}
print(json.dumps(summary, indent=2))
assert len(unique_class) == 184
assert sum(pair_counts.values()) == 430
assert dict(sorted(unique_counts.items())) == {"dead-routed": 21, "legacy-stubbed": 48, "mapped": 115}
assert dict(sorted(pair_counts.items())) == {"dead-routed": 116, "legacy-stubbed": 49, "mapped": 265}
assert len(direct_orphans) == 84
assert len(mixed_aliases) == 23
